# Trend Regime Quadrant Analysis

**두 가지 추세 신호를 활용한 국면 분석**

| | Fast Trend (+) | Fast Trend (-) |
|---|---|---|
| **Slow Trend (+)** | 강세 (Bullish) | 조정 (Correction) |
| **Slow Trend (-)** | 반등 (Recovery) | 약세 (Bearish) |

- **Fast Trend**: 단기 모멘텀 (윈도우: config 셀에서 설정)
- **Slow Trend**: 장기 모멘텀 (윈도우: config 셀에서 설정)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.data import load_price_data
from src.config import TICKER_PRESETS, TICKER_DESCRIPTIONS, DEFAULT_START_DATE

In [ ]:
# ====== 파라미터 설정 (최적화 후 여기만 업데이트) ======
FAST_WINDOW = 3     # Fast trend 윈도우 (월)
SLOW_WINDOW = 12    # Slow trend 윈도우 (월)
BEAR_FLOOR = 0.2    # 약세 국면 최소 비중

# 비중 자동 계산: 강세(1.0) ~ 약세(BEAR_FLOOR) 선형 보간
def make_weights(bear_floor):
    spread = 1.0 - bear_floor
    return {
        '강세 (Bullish)': 1.0,
        '반등 (Recovery)': bear_floor + spread * 2/3,
        '조정 (Correction)': bear_floor + spread * 1/3,
        '약세 (Bearish)': bear_floor,
    }

REGIME_WEIGHTS = make_weights(BEAR_FLOOR)
print(f"Fast={FAST_WINDOW}M, Slow={SLOW_WINDOW}M, Bear Floor={BEAR_FLOOR}")
for k, v in REGIME_WEIGHTS.items():
    print(f"  {k}: {v:.2f}")

In [2]:
tickers = TICKER_PRESETS["Alpha (Default)"]
prices, missing = load_price_data(tickers, start_date=DEFAULT_START_DATE)
print(f"Loaded: {len(prices.columns)} tickers, Missing: {missing}")
print(f"Date range: {prices.index[0].date()} ~ {prices.index[-1].date()}")
prices.tail(3)

Loaded: 24 tickers, Missing: []
Date range: 2000-05-26 ~ 2026-04-01


Ticker,SMH,IGV,XAR,XBI,XME,XOP,PAVE,ARKK,MGK,MGV,...,QUAL,GDX,URA,IXN,372330.KS,283580.KS,IGF,BOTZ,SKYY,ICLN
Date,,,,,,,,,,,,,,,,,,,,,
2026-03-30,362.529999,77.620003,242.240005,118.779999,103.459999,185.479996,49.189999,63.520000,353.489990,142.630005,...,186.490005,85.790001,45.290001,95.760002,7785.0,15850.0,66.470001,31.990000,105.190002,17.510000
2026-03-31,383.399994,80.050003,253.979996,127.730003,108.010002,181.830002,50.810001,67.589996,367.440002,144.949997,...,191.809998,91.769997,48.430000,99.970001,7760.0,15930.0,67.000000,33.220001,109.360001,18.290001
2026-04-01,383.399994,80.050003,253.979996,127.730003,108.010002,181.830002,50.810001,67.589996,367.440002,144.949997,...,191.809998,91.769997,48.430000,99.970001,7840.0,15890.0,67.000000,33.220001,109.360001,18.290001


In [3]:
# 월말 가격으로 리샘플
monthly_prices = prices.resample('BME').last()

# 수익률 계산 (설정된 윈도우 사용)
fast_ret = monthly_prices.pct_change(FAST_WINDOW)
slow_ret = monthly_prices.pct_change(SLOW_WINDOW)

# 국면 매핑
REGIMES = {
    (True, True): '강세 (Bullish)',
    (True, False): '조정 (Correction)',
    (False, True): '반등 (Recovery)',
    (False, False): '약세 (Bearish)',
}

REGIME_COLORS = {
    '강세 (Bullish)': '#2ecc71',
    '조정 (Correction)': '#f39c12',
    '반등 (Recovery)': '#3498db',
    '약세 (Bearish)': '#e74c3c',
}

def classify_regime(slow, fast):
    """Classify regime based on slow and fast trend signals."""
    return REGIMES.get((slow > 0, fast > 0), np.nan)

# 각 티커별 국면 판정
regime_df = pd.DataFrame(index=monthly_prices.index, columns=monthly_prices.columns)
for col in monthly_prices.columns:
    for dt in monthly_prices.index:
        s = slow_ret.loc[dt, col]
        f = fast_ret.loc[dt, col]
        if pd.notna(s) and pd.notna(f):
            regime_df.loc[dt, col] = classify_regime(s, f)

regime_df = regime_df.dropna(how='all')
print(f'Fast={FAST_WINDOW}M, Slow={SLOW_WINDOW}M')
print(f"Regime data: {regime_df.shape[0]} months x {regime_df.shape[1]} tickers")
regime_df.tail(3)

Regime data: 300 months x 24 tickers


Ticker,SMH,IGV,XAR,XBI,XME,XOP,PAVE,ARKK,MGK,MGV,...,QUAL,GDX,URA,IXN,372330.KS,283580.KS,IGF,BOTZ,SKYY,ICLN
Date,,,,,,,,,,,,,,,,,,,,,
2026-02-27,강세 (Bullish),약세 (Bearish),강세 (Bullish),강세 (Bullish),강세 (Bullish),강세 (Bullish),강세 (Bullish),조정 (Correction),조정 (Correction),강세 (Bullish),...,강세 (Bullish),강세 (Bullish),강세 (Bullish),강세 (Bullish),약세 (Bearish),강세 (Bullish),강세 (Bullish),강세 (Bullish),약세 (Bearish),강세 (Bullish)
2026-03-31,강세 (Bullish),약세 (Bearish),강세 (Bullish),강세 (Bullish),강세 (Bullish),강세 (Bullish),강세 (Bullish),조정 (Correction),조정 (Correction),강세 (Bullish),...,조정 (Correction),강세 (Bullish),강세 (Bullish),조정 (Correction),약세 (Bearish),강세 (Bullish),강세 (Bullish),조정 (Correction),조정 (Correction),강세 (Bullish)
2026-04-30,조정 (Correction),약세 (Bearish),조정 (Correction),강세 (Bullish),조정 (Correction),강세 (Bullish),강세 (Bullish),조정 (Correction),조정 (Correction),조정 (Correction),...,조정 (Correction),조정 (Correction),조정 (Correction),조정 (Correction),약세 (Bearish),강세 (Bullish),강세 (Bullish),조정 (Correction),조정 (Correction),강세 (Bullish)


In [4]:
# 최신 시점 국면 현황
latest_date = regime_df.index[-1]
latest_regime = regime_df.loc[latest_date].dropna()
latest_fast = fast_ret.loc[latest_date].dropna()
latest_slow = slow_ret.loc[latest_date].dropna()

# 현재 국면 요약 테이블
summary = pd.DataFrame({
    'Ticker': latest_regime.index,
    'Description': [TICKER_DESCRIPTIONS.get(t, t) for t in latest_regime.index],
    f'Slow ({SLOW_WINDOW}M)': [f"{latest_slow.get(t, 0):.1%}" for t in latest_regime.index],
    f'Fast ({FAST_WINDOW}M)': [f"{latest_fast.get(t, 0):.1%}" for t in latest_regime.index],
    'Regime': latest_regime.values,
}).set_index('Ticker')

print(f"Date: {latest_date.date()}\n")
for regime_name in ['강세 (Bullish)', '조정 (Correction)', '반등 (Recovery)', '약세 (Bearish)']:
    subset = summary[summary['Regime'] == regime_name]
    if len(subset) > 0:
        print(f"{'='*50}")
        print(f" {regime_name}: {len(subset)} tickers")
        print(f"{'='*50}")
        print(subset[['Description', f'Slow ({SLOW_WINDOW}M)', f'Fast ({FAST_WINDOW}M)']].to_string())
        print()

Date: 2026-04-30

 강세 (Bullish): 7 tickers
                     Description Slow (12M) Fast (1M)
Ticker                                               
XBI                      Biotech      54.6%      2.4%
XOP                Oil & Gas E&P      68.2%     30.2%
PAVE              Infrastructure      33.6%      0.1%
SCHD                    Dividend      23.4%      3.8%
283580.KS          China CSI 300      36.7%      1.2%
IGF        Global Infrastructure      22.8%      4.0%
ICLN                Clean Energy      58.3%      0.8%

 조정 (Correction): 15 tickers
                Description Slow (12M) Fast (1M)
Ticker                                          
SMH          Semiconductors      82.0%     -5.0%
XAR     Aerospace & Defense      49.1%     -6.9%
XME         Metals & Mining      92.5%     -9.0%
ARKK             Innovation      32.9%     -9.7%
MGK         Mega Cap Growth      16.9%     -9.4%
MGV          Mega Cap Value      19.7%     -1.7%
IWM               Small Cap      28.6%     -4.3%


In [5]:
# 사분면 산점도: X = Slow Trend, Y = Fast Trend
# 표준 사분면: 1(우상)=강세, 2(좌상)=반등, 3(좌하)=약세, 4(우하)=조정

TARGET_DATE = None  # 특정 날짜를 보려면 '2024-06-30' 등 입력, None이면 최신

if TARGET_DATE:
    target_ts = pd.Timestamp(TARGET_DATE)
    valid_dates = regime_df.index[regime_df.index <= target_ts]
    if len(valid_dates) == 0:
        raise ValueError(f'{TARGET_DATE} 이전 데이터가 없습니다.')
    plot_date = valid_dates[-1]
else:
    plot_date = regime_df.index[-1]

plot_regime = regime_df.loc[plot_date].dropna()
plot_fast = fast_ret.loc[plot_date].dropna()
plot_slow = slow_ret.loc[plot_date].dropna()

slow_label = f'Slow ({SLOW_WINDOW}M Return)'
fast_label = f'Fast ({FAST_WINDOW}M Return)'

scatter_data = pd.DataFrame({
    slow_label: plot_slow,
    fast_label: plot_fast,
    'Regime': plot_regime,
}).dropna()
scatter_data['Description'] = [TICKER_DESCRIPTIONS.get(t, t) for t in scatter_data.index]

fig = go.Figure()

# 모든 국면을 항상 범례에 표시
for regime_name, color in REGIME_COLORS.items():
    mask = scatter_data['Regime'] == regime_name
    subset = scatter_data[mask]
    fig.add_trace(go.Scatter(
        x=subset[slow_label] if len(subset) > 0 else [None],
        y=subset[fast_label] if len(subset) > 0 else [None],
        mode='markers+text',
        name=regime_name,
        text=subset['Description'] if len(subset) > 0 else [None],
        textposition='top center',
        textfont=dict(size=9),
        marker=dict(size=12, color=color, line=dict(width=1, color='white')),
        hovertemplate=f'%{{text}}<br>{SLOW_WINDOW}M: %{{x:.1%}}<br>{FAST_WINDOW}M: %{{y:.1%}}<extra></extra>',
    ))

# 축선
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.5)

# 사분면 라벨
x_range = max(abs(scatter_data[slow_label].min()), abs(scatter_data[slow_label].max()))
y_range = max(abs(scatter_data[fast_label].min()), abs(scatter_data[fast_label].max()))

annotations = [
    dict(x=x_range*0.7,  y=y_range*0.8,  text='강세', font=dict(size=28, color='rgba(46,204,113,0.5)'), showarrow=False),
    dict(x=-x_range*0.7, y=y_range*0.8,  text='반등', font=dict(size=28, color='rgba(52,152,219,0.5)'), showarrow=False),
    dict(x=x_range*0.7,  y=-y_range*0.8, text='조정', font=dict(size=28, color='rgba(243,156,18,0.5)'), showarrow=False),
    dict(x=-x_range*0.7, y=-y_range*0.8, text='약세', font=dict(size=28, color='rgba(231,76,60,0.5)'), showarrow=False),
]

fig.update_layout(
    title=f'Trend Regime Quadrant ({plot_date.date()})',
    xaxis_title=f'← Slow Trend ({SLOW_WINDOW}M Return) →',
    yaxis_title=f'← Fast Trend ({FAST_WINDOW}M Return) →',
    xaxis=dict(tickformat='.0%', zeroline=True, title_font=dict(size=16)),
    yaxis=dict(tickformat='.0%', zeroline=True, title_font=dict(size=16)),
    annotations=annotations,
    template='plotly_white',
    height=700,
    width=700,
)
fig.show()
print(f'Date: {plot_date.date()}')

In [6]:
# 국면을 숫자로 인코딩 (히트맵용)
regime_encoding = {
    '강세 (Bullish)': 3,
    '반등 (Recovery)': 2,
    '조정 (Correction)': 1,
    '약세 (Bearish)': 0,
}
regime_numeric = regime_df.replace(regime_encoding).apply(pd.to_numeric, errors='coerce')

# 최근 24개월만 표시
recent = regime_numeric.tail(24).T
recent.index = [TICKER_DESCRIPTIONS.get(t, t) for t in recent.index]
recent.columns = [d.strftime('%Y-%m') for d in regime_numeric.tail(24).index]

# 커스텀 컬러스케일
colorscale = [
    [0.0, '#e74c3c'],    # 약세
    [0.333, '#f39c12'],  # 조정
    [0.667, '#3498db'],  # 반등
    [1.0, '#2ecc71'],    # 강세
]

# 히트맵용 텍스트 (국면명 약어)
regime_text_map = {3: '강세', 2: '반등', 1: '조정', 0: '약세'}
text_matrix = regime_df.tail(24).replace(
    {v: k for k, v in regime_encoding.items()}
).T
text_matrix.index = recent.index
text_matrix.columns = recent.columns
# 짧은 라벨로 변환
text_short = text_matrix.replace({
    '강세 (Bullish)': '강',
    '조정 (Correction)': '조',
    '반등 (Recovery)': '반',
    '약세 (Bearish)': '약',
})

fig = go.Figure(data=go.Heatmap(
    z=recent.values,
    x=recent.columns,
    y=recent.index,
    text=text_short.values,
    texttemplate='%{text}',
    textfont=dict(size=10),
    colorscale=colorscale,
    zmin=0, zmax=3,
    colorbar=dict(
        tickvals=[0, 1, 2, 3],
        ticktext=['약세', '조정', '반등', '강세'],
    ),
    hovertemplate='%{y}<br>%{x}<br>%{text}<extra></extra>',
))

fig.update_layout(
    title='Regime History Heatmap (Last 24 Months)',
    xaxis_title='Month',
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    height=max(500, len(recent) * 28),
    width=1100,
)
fig.show()

In [7]:
def regime_backtest(prices, regime_df, regime_weights, tcost=0.002, backtest_start='2015-01-01'):
    """
    국면 기반 비중 차등 전략 백테스트.
    
    월말 국면 판단 → 다음달 비중 적용 (implementation lag).
    비중은 국면별 raw weight를 정규화하여 합이 1이 되도록 함.
    투자할 티커가 없으면(모두 약세) 현금 보유.
    """
    monthly_prices = prices.resample('BME').last()
    monthly_ret = monthly_prices.pct_change(1).shift(-1)  # 다음 달 수익률
    
    # 국면 기반 raw weight
    raw_weights = regime_df.replace(regime_weights)
    raw_weights = raw_weights.apply(pd.to_numeric, errors='coerce')
    
    # 정규화: 합이 1이 되도록
    weight_sum = raw_weights.sum(axis=1)
    weights = raw_weights.div(weight_sum, axis=0).fillna(0)
    
    # backtest 시작일 필터
    mask = weights.index >= pd.Timestamp(backtest_start)
    weights = weights.loc[mask]
    monthly_ret = monthly_ret.reindex(weights.index)
    
    # 공통 컬럼
    common_cols = weights.columns.intersection(monthly_ret.columns)
    weights = weights[common_cols]
    monthly_ret = monthly_ret[common_cols].fillna(0)
    
    # 거래비용: 비중 변화의 절대값 합 * tcost
    weight_diff = weights.diff().abs().sum(axis=1).fillna(0)
    
    # 포트폴리오 수익률
    port_ret = (weights * monthly_ret).sum(axis=1) - weight_diff * tcost
    
    # NAV
    nav = (1 + port_ret).cumprod()
    nav = nav.dropna()
    
    return nav, weights, port_ret


# 벤치마크: 동일가중 (국면 무관)
def equal_weight_backtest(prices, regime_df, tcost=0.002, backtest_start='2015-01-01'):
    """동일가중 벤치마크."""
    monthly_prices = prices.resample('BME').last()
    monthly_ret = monthly_prices.pct_change(1).shift(-1)
    
    # 데이터 있는 티커에 동일 비중
    available = regime_df.notna().astype(float)
    n_available = available.sum(axis=1)
    weights = available.div(n_available, axis=0).fillna(0)
    
    mask = weights.index >= pd.Timestamp(backtest_start)
    weights = weights.loc[mask]
    monthly_ret = monthly_ret.reindex(weights.index)
    
    common_cols = weights.columns.intersection(monthly_ret.columns)
    weights = weights[common_cols]
    monthly_ret = monthly_ret[common_cols].fillna(0)
    
    weight_diff = weights.diff().abs().sum(axis=1).fillna(0)
    port_ret = (weights * monthly_ret).sum(axis=1) - weight_diff * tcost
    
    nav = (1 + port_ret).cumprod().dropna()
    return nav, port_ret


# 실행 (상단 config의 REGIME_WEIGHTS 사용)
nav_regime, weights_regime, ret_regime = regime_backtest(prices, regime_df, REGIME_WEIGHTS)
nav_ew, ret_ew = equal_weight_backtest(prices, regime_df)

print(f"Backtest period: {nav_regime.index[0].date()} ~ {nav_regime.index[-1].date()}")
print(f"Regime weights: {REGIME_WEIGHTS}")

Backtest period: 2015-01-30 ~ 2026-04-30
Regime weights: {'강세 (Bullish)': 1.0, '반등 (Recovery)': 0.5, '조정 (Correction)': 0.25, '약세 (Bearish)': 0.0}


In [8]:
def calc_stats(nav, ret, name):
    """성과 지표 계산."""
    years = (nav.index[-1] - nav.index[0]).days / 365.25
    cagr = (nav.iloc[-1] / nav.iloc[0]) ** (1 / years) - 1
    vol = ret.std() * np.sqrt(12)  # 월간 → 연율
    sharpe = cagr / vol if vol > 0 else 0
    
    # MDD
    cummax = nav.cummax()
    dd = (nav - cummax) / cummax
    mdd = dd.min()
    
    return {
        'Strategy': name,
        'CAGR': f"{cagr:.1%}",
        'Volatility': f"{vol:.1%}",
        'Sharpe': f"{sharpe:.2f}",
        'MDD': f"{mdd:.1%}",
        'Final NAV': f"{nav.iloc[-1]:.2f}",
    }

stats = pd.DataFrame([
    calc_stats(nav_regime, ret_regime, 'Regime-Based'),
    calc_stats(nav_ew, ret_ew, 'Equal Weight (BM)'),
]).set_index('Strategy')

print(stats.to_string())
print(f"\nRegime weights: 강세={REGIME_WEIGHTS['강세 (Bullish)']}, "
      f"반등={REGIME_WEIGHTS['반등 (Recovery)']}, "
      f"조정={REGIME_WEIGHTS['조정 (Correction)']}, "
      f"약세={REGIME_WEIGHTS['약세 (Bearish)']}")

                    CAGR Volatility Sharpe     MDD Final NAV
Strategy                                                    
Regime-Based       12.1%      16.8%   0.72  -31.2%      3.82
Equal Weight (BM)  13.1%      16.9%   0.78  -25.5%      4.25

Regime weights: 강세=1.0, 반등=0.5, 조정=0.25, 약세=0.0


In [9]:
# NAV 비교 차트
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['NAV Comparison', 'Drawdown'],
                    row_heights=[0.65, 0.35], vertical_spacing=0.08)

# NAV
fig.add_trace(go.Scatter(
    x=nav_regime.index, y=nav_regime.values,
    name='Regime-Based', line=dict(color='#2ecc71', width=2),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=nav_ew.index, y=nav_ew.values,
    name='Equal Weight (BM)', line=dict(color='gray', width=1.5, dash='dash'),
), row=1, col=1)

# Drawdown
for nav, name, color, dash in [
    (nav_regime, 'Regime-Based', '#2ecc71', None),
    (nav_ew, 'Equal Weight (BM)', 'gray', 'dash'),
]:
    dd = (nav - nav.cummax()) / nav.cummax()
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd.values,
        name=name, line=dict(color=color, width=1.5, dash=dash),
        showlegend=False, fill='tozeroy', fillcolor=f'rgba({",".join(str(int(color.lstrip("#")[i:i+2], 16)) for i in (0,2,4))},0.1)' if color != 'gray' else 'rgba(128,128,128,0.1)',
    ), row=2, col=1)

fig.update_yaxes(tickformat='.0%', row=2, col=1)
fig.update_layout(
    template='plotly_white',
    height=600, width=900,
    title='Regime-Based Strategy vs Equal Weight Benchmark',
)
fig.show()

## 8. 국면별 비중 민감도 분석

여러 비중 조합으로 성과가 어떻게 달라지는지 확인

In [10]:
# 다양한 비중 조합 테스트
weight_scenarios = {
    'Aggressive':    {'강세 (Bullish)': 1.0, '반등 (Recovery)': 0.7, '조정 (Correction)': 0.3, '약세 (Bearish)': 0.0},
    'Moderate':      {'강세 (Bullish)': 1.0, '반등 (Recovery)': 0.5, '조정 (Correction)': 0.25, '약세 (Bearish)': 0.0},
    'Conservative':  {'강세 (Bullish)': 1.0, '반등 (Recovery)': 0.3, '조정 (Correction)': 0.1, '약세 (Bearish)': 0.0},
    'Binary':        {'강세 (Bullish)': 1.0, '반등 (Recovery)': 0.0, '조정 (Correction)': 0.0, '약세 (Bearish)': 0.0},
    'No Bearish':    {'강세 (Bullish)': 1.0, '반등 (Recovery)': 1.0, '조정 (Correction)': 1.0, '약세 (Bearish)': 0.0},
}

results = []
navs = {}

for scenario_name, weights in weight_scenarios.items():
    nav, _, ret = regime_backtest(prices, regime_df, weights)
    navs[scenario_name] = nav
    years = (nav.index[-1] - nav.index[0]).days / 365.25
    cagr = (nav.iloc[-1] / nav.iloc[0]) ** (1 / years) - 1
    vol = ret.std() * np.sqrt(12)
    sharpe = cagr / vol if vol > 0 else 0
    mdd = ((nav - nav.cummax()) / nav.cummax()).min()
    results.append({
        'Scenario': scenario_name,
        'Weights (강/반/조/약)': f"{weights['강세 (Bullish)']}/{weights['반등 (Recovery)']}/{weights['조정 (Correction)']}/{weights['약세 (Bearish)']}",
        'CAGR': f"{cagr:.1%}",
        'Vol': f"{vol:.1%}",
        'Sharpe': f"{sharpe:.2f}",
        'MDD': f"{mdd:.1%}",
    })

# EW 벤치마크 추가
years = (nav_ew.index[-1] - nav_ew.index[0]).days / 365.25
cagr_ew = (nav_ew.iloc[-1] / nav_ew.iloc[0]) ** (1 / years) - 1
vol_ew = ret_ew.std() * np.sqrt(12)
results.append({
    'Scenario': 'Equal Weight (BM)',
    'Weights (강/반/조/약)': '1/1/1/1',
    'CAGR': f"{cagr_ew:.1%}",
    'Vol': f"{vol_ew:.1%}",
    'Sharpe': f"{cagr_ew/vol_ew:.2f}",
    'MDD': f"{((nav_ew - nav_ew.cummax()) / nav_ew.cummax()).min():.1%}",
})

pd.DataFrame(results).set_index('Scenario')

,Weights (강/반/조/약),CAGR,Vol,Sharpe,MDD
Scenario,,,,,
Aggressive,1.0/0.7/0.3/0.0,12.0%,16.7%,0.71,-31.6%
Moderate,1.0/0.5/0.25/0.0,12.1%,16.8%,0.72,-31.2%
Conservative,1.0/0.3/0.1/0.0,11.7%,17.0%,0.69,-32.8%
Binary,1.0/0.0/0.0/0.0,8.9%,17.1%,0.52,-31.8%
No Bearish,1.0/1.0/1.0/0.0,12.7%,16.7%,0.76,-28.5%
Equal Weight (BM),1/1/1/1,13.1%,16.9%,0.78,-25.5%


In [11]:
# 시나리오별 NAV 비교 차트
fig = go.Figure()

colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12']
for (name, nav), color in zip(navs.items(), colors):
    fig.add_trace(go.Scatter(
        x=nav.index, y=nav.values,
        name=name, line=dict(color=color, width=2),
    ))

fig.add_trace(go.Scatter(
    x=nav_ew.index, y=nav_ew.values,
    name='Equal Weight (BM)', line=dict(color='gray', width=1.5, dash='dash'),
))

fig.update_layout(
    title='Regime-Based Strategy: Weight Sensitivity Analysis',
    yaxis_title='NAV',
    template='plotly_white',
    height=500, width=900,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

In [12]:
# 국면별 티커 수 비율 시계열
regime_counts = pd.DataFrame(index=regime_df.index)
for regime_name in ['강세 (Bullish)', '반등 (Recovery)', '조정 (Correction)', '약세 (Bearish)']:
    regime_counts[regime_name] = (regime_df == regime_name).sum(axis=1)

total = regime_counts.sum(axis=1)
regime_pct = regime_counts.div(total, axis=0)

fig = go.Figure()
for regime_name, color in REGIME_COLORS.items():
    fig.add_trace(go.Scatter(
        x=regime_pct.index,
        y=regime_pct[regime_name],
        name=regime_name,
        stackgroup='one',
        fillcolor=color,
        line=dict(width=0.5, color=color),
    ))

fig.update_layout(
    title='Regime Distribution Over Time (% of Universe)',
    yaxis=dict(tickformat='.0%', title='Proportion'),
    template='plotly_white',
    height=400, width=900,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

## 10. 파라미터 최적화

그리드 서치로 최적 파라미터를 탐색합니다:
- **Fast window**: 1, 2, 3, 6개월
- **Slow window**: 6, 9, 12개월
- **약세 최소 비중**: 0, 0.1, 0.2, 0.3
- **비중 스프레드**: 강세~약세 간 비중 차이 구조

In-sample (2015~2020) / Out-of-sample (2021~) 분할로 오버피팅을 검증합니다.

In [13]:
from itertools import product

def build_regime_df(prices, fast_window, slow_window):
    """주어진 윈도우로 국면 DataFrame 생성."""
    mp = prices.resample('BME').last()
    f_ret = mp.pct_change(fast_window)
    s_ret = mp.pct_change(slow_window)
    
    rdf = pd.DataFrame(index=mp.index, columns=mp.columns)
    for col in mp.columns:
        for dt in mp.index:
            s, f = s_ret.loc[dt, col], f_ret.loc[dt, col]
            if pd.notna(s) and pd.notna(f):
                rdf.loc[dt, col] = classify_regime(s, f)
    return rdf.dropna(how='all'), f_ret, s_ret


def regime_backtest_stats(prices, rdf, regime_weights, tcost=0.002, start='2015-01-01', end=None):
    """백테스트 후 Sharpe 등 통계 반환."""
    mp = prices.resample('BME').last()
    m_ret = mp.pct_change(1).shift(-1)
    
    raw_w = rdf.replace(regime_weights).apply(pd.to_numeric, errors='coerce')
    w_sum = raw_w.sum(axis=1)
    w = raw_w.div(w_sum, axis=0).fillna(0)
    
    mask = w.index >= pd.Timestamp(start)
    if end:
        mask &= w.index < pd.Timestamp(end)
    w = w.loc[mask]
    m_ret = m_ret.reindex(w.index)
    
    cc = w.columns.intersection(m_ret.columns)
    w, m_ret = w[cc], m_ret[cc].fillna(0)
    
    tc = w.diff().abs().sum(axis=1).fillna(0)
    p_ret = (w * m_ret).sum(axis=1) - tc * tcost
    nav = (1 + p_ret).cumprod().dropna()
    
    if len(nav) < 12:
        return None
    
    years = (nav.index[-1] - nav.index[0]).days / 365.25
    cagr = (nav.iloc[-1] / nav.iloc[0]) ** (1 / years) - 1
    vol = p_ret.std() * np.sqrt(12)
    sharpe = cagr / vol if vol > 0 else 0
    mdd = ((nav - nav.cummax()) / nav.cummax()).min()
    turnover = tc.mean() * 12  # 연간 턴오버
    
    return {'cagr': cagr, 'vol': vol, 'sharpe': sharpe, 'mdd': mdd, 'turnover': turnover, 'nav': nav}


# 그리드 서치 파라미터
fast_windows = [1, 2, 3, 6]
slow_windows = [6, 9, 12]
bearish_floors = [0.0, 0.1, 0.2, 0.3]

# make_weights()는 상단 config 셀에 정의됨

IS_END = '2021-01-01'  # In-sample 끝

print("Running grid search...")
print(f"Combinations: {len(fast_windows) * len(slow_windows) * len(bearish_floors)}")

grid_results = []
regime_cache = {}

for fw, sw, bf in product(fast_windows, slow_windows, bearish_floors):
    if fw >= sw:  # fast는 slow보다 짧아야 함
        continue
    
    # 캐시: 같은 윈도우 조합은 regime_df 재사용
    cache_key = (fw, sw)
    if cache_key not in regime_cache:
        regime_cache[cache_key] = build_regime_df(prices, fw, sw)
    rdf, _, _ = regime_cache[cache_key]
    
    rw = make_weights(bf)
    
    # In-sample
    is_stats = regime_backtest_stats(prices, rdf, rw, start='2015-01-01', end=IS_END)
    # Out-of-sample
    oos_stats = regime_backtest_stats(prices, rdf, rw, start=IS_END)
    
    if is_stats and oos_stats:
        grid_results.append({
            'Fast': fw, 'Slow': sw, 'Bear Floor': bf,
            'Weights': f"{rw['강세 (Bullish)']:.1f}/{rw['반등 (Recovery)']:.2f}/{rw['조정 (Correction)']:.2f}/{rw['약세 (Bearish)']:.1f}",
            'IS Sharpe': is_stats['sharpe'],
            'IS CAGR': is_stats['cagr'],
            'IS MDD': is_stats['mdd'],
            'OOS Sharpe': oos_stats['sharpe'],
            'OOS CAGR': oos_stats['cagr'],
            'OOS MDD': oos_stats['mdd'],
            'Turnover': is_stats['turnover'],
        })

grid_df = pd.DataFrame(grid_results)
print(f"Valid combinations: {len(grid_df)}")

# IS Sharpe 기준 상위 10개
top10 = grid_df.sort_values('IS Sharpe', ascending=False).head(10).copy()
for col in ['IS Sharpe', 'OOS Sharpe']:
    top10[col] = top10[col].map('{:.2f}'.format)
for col in ['IS CAGR', 'OOS CAGR', 'IS MDD', 'OOS MDD']:
    top10[col] = top10[col].map('{:.1%}'.format)
top10['Turnover'] = top10['Turnover'].map('{:.0%}'.format)

print("\n=== Top 10 by In-Sample Sharpe ===")
print(top10.to_string(index=False))

Running grid search...
Combinations: 48
Valid combinations: 44

=== Top 10 by In-Sample Sharpe ===
 Fast  Slow  Bear Floor           Weights IS Sharpe IS CAGR IS MDD OOS Sharpe OOS CAGR OOS MDD Turnover
    6    12         0.0 1.0/0.67/0.33/0.0      0.95   17.3% -18.7%       0.72    12.9%  -21.5%     387%
    6     9         0.0 1.0/0.67/0.33/0.0      0.93   17.8% -19.3%       0.73    13.7%  -21.5%     483%
    6    12         0.1 1.0/0.70/0.40/0.1      0.93   16.5% -19.2%       0.70    11.4%  -22.3%     304%
    6     9         0.1 1.0/0.70/0.40/0.1      0.92   16.4% -19.7%       0.71    11.4%  -22.0%     344%
    6    12         0.2 1.0/0.73/0.47/0.2      0.91   16.0% -19.7%       0.71    11.3%  -22.8%     249%
    6     9         0.2 1.0/0.73/0.47/0.2      0.90   15.9% -20.1%       0.71    11.3%  -22.6%     274%
    3     6         0.1 1.0/0.70/0.40/0.1      0.90   15.8% -20.2%       0.59     9.1%  -27.8%     415%
    6    12         0.3 1.0/0.77/0.53/0.3      0.89   15.7% -20.2%   

### IS vs OOS Sharpe 산점도

오버피팅 여부를 시각적으로 확인합니다. 대각선에 가까울수록 IS/OOS 성과가 일관적입니다.

In [14]:
# IS vs OOS Sharpe 산점도
fig = go.Figure()

for bf in bearish_floors:
    subset = grid_df[grid_df['Bear Floor'] == bf]
    fig.add_trace(go.Scatter(
        x=subset['IS Sharpe'], y=subset['OOS Sharpe'],
        mode='markers',
        name=f'Bear Floor={bf}',
        marker=dict(size=10, opacity=0.7),
        text=[f"F{r['Fast']}/S{r['Slow']}" for _, r in subset.iterrows()],
        hovertemplate='%{text}<br>IS: %{x:.2f}<br>OOS: %{y:.2f}<extra></extra>',
    ))

# 대각선
sharpe_range = [min(grid_df['IS Sharpe'].min(), grid_df['OOS Sharpe'].min()),
                max(grid_df['IS Sharpe'].max(), grid_df['OOS Sharpe'].max())]
fig.add_trace(go.Scatter(
    x=sharpe_range, y=sharpe_range,
    mode='lines', line=dict(dash='dash', color='gray'),
    showlegend=False,
))

fig.update_layout(
    title='In-Sample vs Out-of-Sample Sharpe Ratio',
    xaxis_title='In-Sample Sharpe (2015~2020)',
    yaxis_title='Out-of-Sample Sharpe (2021~)',
    template='plotly_white',
    height=500, width=600,
)
fig.show()

### 최적 파라미터 vs 기존 전략 vs BM 비교

IS Sharpe 상위 중 OOS에서도 안정적인 파라미터를 선정하여 비교합니다.

In [15]:
# IS+OOS 합산 Sharpe 기준 Best 선정 (오버피팅 방지)
grid_df['Combined Sharpe'] = grid_df['IS Sharpe'] * 0.5 + grid_df['OOS Sharpe'] * 0.5
best = grid_df.sort_values('Combined Sharpe', ascending=False).iloc[0]

best_fw, best_sw, best_bf = int(best['Fast']), int(best['Slow']), best['Bear Floor']
best_rw = make_weights(best_bf)

print(f"Best params: Fast={best_fw}M, Slow={best_sw}M, Bear Floor={best_bf}")
print(f"Weights: 강세={best_rw['강세 (Bullish)']:.2f}, 반등={best_rw['반등 (Recovery)']:.2f}, "
      f"조정={best_rw['조정 (Correction)']:.2f}, 약세={best_rw['약세 (Bearish)']:.2f}")
print(f"IS Sharpe={best['IS Sharpe']:.2f}, OOS Sharpe={best['OOS Sharpe']:.2f}")

# 최적 파라미터로 전체 기간 백테스트
best_rdf, _, _ = regime_cache[(best_fw, best_sw)]
nav_best, _, ret_best = regime_backtest(prices, best_rdf, best_rw)

# 현재 config 기준
config_label = f'Config (F{FAST_WINDOW}/S{SLOW_WINDOW}/BF{BEAR_FLOOR})'

# 비교 차트
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['NAV Comparison', 'Drawdown'],
                    row_heights=[0.65, 0.35], vertical_spacing=0.08)

traces = [
    (nav_best, f'Grid Best (F{best_fw}/S{best_sw}/BF{best_bf})', '#2ecc71', None),
    (nav_regime, config_label, '#e74c3c', 'dash'),
    (nav_ew, 'Equal Weight (BM)', 'gray', 'dot'),
]

for nav, name, color, dash in traces:
    fig.add_trace(go.Scatter(
        x=nav.index, y=nav.values,
        name=name, line=dict(color=color, width=2, dash=dash),
    ), row=1, col=1)
    
    dd = (nav - nav.cummax()) / nav.cummax()
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd.values,
        name=name, line=dict(color=color, width=1.5, dash=dash),
        showlegend=False, fill='tozeroy',
        fillcolor=f'rgba({",".join(str(int(color.lstrip("#")[i:i+2], 16)) for i in (0,2,4))},0.08)' if color != 'gray' else 'rgba(128,128,128,0.08)',
    ), row=2, col=1)

# IS/OOS 경계선
is_end_str = IS_END
fig.add_shape(type='line', x0=is_end_str, x1=is_end_str, y0=0, y1=1,
              yref='paper', line=dict(dash='dash', color='orange', width=1.5),
              opacity=0.6)
fig.add_annotation(x=is_end_str, y=1.02, yref='paper', text='IS / OOS',
                   showarrow=False, font=dict(color='orange', size=11))

fig.update_yaxes(tickformat='.0%', row=2, col=1)
fig.update_layout(template='plotly_white', height=600, width=900,
                  title='Grid Best vs Config vs Benchmark')
fig.show()

# 통계 비교
print("\n=== Performance Comparison (Full Period) ===")
stats_compare = pd.DataFrame([
    calc_stats(nav_best, ret_best, f'Grid Best (F{best_fw}/S{best_sw}/BF{best_bf})'),
    calc_stats(nav_regime, ret_regime, config_label),
    calc_stats(nav_ew, ret_ew, 'Equal Weight (BM)'),
]).set_index('Strategy')
print(stats_compare.to_string())

Best params: Fast=6M, Slow=12M, Bear Floor=0.0
Weights: 강세=1.00, 반등=0.67, 조정=0.33, 약세=0.00
IS Sharpe=0.95, OOS Sharpe=0.72



=== Performance Comparison (Full Period) ===
                           CAGR Volatility Sharpe     MDD Final NAV
Strategy                                                           
Optimized (F6/S12/BF0.0)  15.5%      18.1%   0.85  -21.5%      5.37
Original (F1/S12/BF0)     12.1%      16.8%   0.72  -31.2%      3.82
Equal Weight (BM)         13.1%      16.9%   0.78  -25.5%      4.25
